# Day 4 — LangGraph Integration

## Routing Between Chat, Retrieval & Prediction

This notebook integrates the AFL chat system, retrieval tools, and prediction models using LangGraph.

The system receives a user query and first determines its intent. The query is then routed to the appropriate path:

1. Factual AFL question → Direct Answer
2. AFL statistics or historical data → Retrieval Tool
3. Match or player prediction → Prediction Tool
4. Unsupported or unrelated question → Refusal

After retrieval or prediction, the result is validated before the final response is formatted.

The main goal is to create an explicit and traceable workflow where different types of AFL requests are handled by specialized nodes instead of using one generic processing path.

## Task 1 — Graph Design for the Full System

The first step is to design a LangGraph workflow that connects the existing AFL chat system, retrieval functionality, and prediction models.

The graph uses a shared state object to carry the user's query, conversation history, detected intent, tool results, validation information, errors, clarification requirements, and final response.

The workflow begins with an intent classification and routing node. Based on the detected intent, the query is sent to one of four branches: direct factual answering, AFL data retrieval, prediction, or an off-topic refusal path.

Retrieval and prediction results are passed through a validation node before reaching the final response-formatting node. This provides a clear separation between deciding what should happen, executing the required operation, checking its result, and generating the final response.

This explicit graph structure is useful because prediction responses require special handling. Predictions should be presented as probabilistic model outputs rather than certain future outcomes, while factual and retrieval responses should remain grounded in the available AFL data.
## State Schema

The LangGraph state represents all information that may be required while processing a user request.

The state contains the original user query and conversation history so that the system can support multi-turn interactions. The detected intent records whether the request is factual, retrieval-based, predictive, or off-topic.

Tool results store outputs returned by retrieval or prediction functions. Error and clarification fields allow the graph to handle invalid requests without crashing.

The trace field records the nodes visited during execution. This makes the workflow easier to debug and allows representative state traces to be inspected during end-to-end testing.

In [2]:
from typing import TypedDict, List, Dict, Any, Optional

In [3]:
class AFLState(TypedDict, total=False):
    user_query: str
    conversation_history: List[Dict[str, str]]
    detected_intent: str
    tool_results: Dict[str, Any]
    final_response: str
    error: Optional[str]
    needs_clarification: bool
    clarification_question: str
    trace: List[Dict[str, Any]]

## State Fields

The `user_query` field stores the current request from the user.

The `conversation_history` field stores previous user and assistant messages so that the system can maintain conversational context.

The `detected_intent` field stores the category assigned by the router. The supported categories are `factual`, `retrieval`, `prediction`, and `off-topic`.

The `tool_results` field stores results returned by retrieval or prediction operations.

The `final_response` field stores the response that will ultimately be shown to the user.

The `error` field records execution errors when a node or tool fails.

The `needs_clarification` field indicates that the system requires additional information from the user.

The `clarification_question` field stores the question that should be presented when required information is missing.

The `trace` field records the sequence of nodes and important events during graph execution.

## LangGraph Architecture

The complete workflow follows this structure:

                     User Query
                         ↓
             Intent Classification / Router
                         ↓
       ┌──────────────┬───────────────┬───────────────┬──────────────┐
       ↓              ↓               ↓               ↓
      Factual      Retrieval       Prediction      Off-topic
       ↓              ↓               ↓               ↓
    Direct        Retrieval       Prediction      Refusal
    Answer          Tool            Tool
                     │               │
                     └───────┬───────┘
                             ↓
                         Validation
                             ↓
                     Response Formatting
                             ↓
                        Final Response

The router decides which specialized branch should process the request. Retrieval and prediction operations are validated before the final response is generated.

## Why Explicit LangGraph Routing?

An explicit LangGraph workflow separates different responsibilities into dedicated nodes.

A factual AFL question does not need a prediction model, while a statistical question should use the available retrieval functions. A prediction request requires access to the prediction model and must be formatted as a probabilistic result.

Using explicit routing makes these differences visible in the graph. It also makes the system easier to debug because the exact path taken by a query can be inspected through the state trace.

A single generic agent could potentially decide which tool to use dynamically, but explicit routing provides stronger control over which operations are allowed for each intent. This is especially useful for the prediction path because prediction results need probability or confidence information and must not be presented as guaranteed outcomes.

The graph therefore separates intent detection, tool execution, validation, and response formatting rather than placing all responsibilities inside one large agent.

In [5]:
import sys

!{sys.executable} -m pip install -qU langgraph

In [6]:
from langgraph.graph import StateGraph, START, END

print("LangGraph imported successfully.")

LangGraph imported successfully.


## Intent Classification and Router Node

The router is responsible for identifying the purpose of the user's query.

The system uses four intent categories:

`prediction` is used for questions asking who will win, who is likely to top-score, or other predictive requests.

`retrieval` is used when the user asks for statistics, records, match results, player statistics, or other information that should come from the AFL datasets.

`factual` is used for general AFL questions such as rules, concepts, history, and explanations.

`off-topic` is used for requests outside the supported AFL domain.

In [7]:
import re

PREDICTION_PATTERNS = [
    r"\bwho will win\b",
    r"\bwho'll win\b",
    r"\bwho wins\b",
    r"\bpredict\b",
    r"\bprediction\b",
    r"\btop[- ]?score\b",
    r"\btop player\b",
    r"\bhighest scorer\b",
    r"\bmost disposals\b",
    r"\bwin.*vs\b",
    r"\bvs.*win\b",
    r"\bbeat\b.*\bthis week\b",
]

RETRIEVAL_PATTERNS = [
    r"\bstats?\b",
    r"\bstatistics\b",
    r"\bdisposals\b",
    r"\bgoals\b",
    r"\btackles\b",
    r"\bmarks\b",
    r"\bhow many\b",
    r"\brecord\b",
    r"\bresults?\b",
    r"\blast round\b",
    r"\bseason stats?\b",
]

FACTUAL_PATTERNS = [
    r"\bwhat is afl\b",
    r"\bwhat are the rules\b",
    r"\bhow does afl work\b",
    r"\bexplain afl\b",
    r"\bwhat is a goal\b",
    r"\bwhat is a behind\b",
    r"\bafl history\b",
    r"\baustralian rules football\b",
]

OFF_TOPIC_PATTERNS = [
    r"\bnba\b",
    r"\bnfl\b",
    r"\bcricket\b",
    r"\bsoccer\b",
    r"\bfootball\b",
    r"\btennis\b",
    r"\bformula 1\b",
    r"\bf1\b",
    r"\bweather\b",
    r"\bpython\b",
    r"\bjavascript\b",
    r"\bcapital of\b",
    r"\bjoke\b",
]

In [8]:
def matches_any(query, patterns):
    query = query.lower()

    for pattern in patterns:
        if re.search(pattern, query):
            return True

    return False


def classify_intent(query) -> str:

    query = str(query).strip()

    if matches_any(query, OFF_TOPIC_PATTERNS):
        return "off-topic"

    if matches_any(query, PREDICTION_PATTERNS):
        return "prediction"

    if matches_any(query, RETRIEVAL_PATTERNS):
        return "retrieval"

    if matches_any(query, FACTUAL_PATTERNS):
        return "factual"

    afl_terms = [
        "afl",
        "australian rules",
        "player",
        "team",
        "match",
        "season",
        "round",
        "club"
    ]

    if any(term in query.lower() for term in afl_terms):
        return "factual"

    return "off-topic"

In [10]:
test_queries = [
    "Who will win Collingwood vs Geelong?",
    "What were Geelong's stats last round?",
    "What are the basic rules of AFL?",
    "Who won the latest NBA game?",
    "Who will top-score for Richmond?"
]

for query in test_queries:
    print(f"Query: {query}")
    print(f"Intent: {classify_intent(query)}")
    print("-" * 60)

Query: Who will win Collingwood vs Geelong?
Intent: prediction
------------------------------------------------------------
Query: What were Geelong's stats last round?
Intent: retrieval
------------------------------------------------------------
Query: What are the basic rules of AFL?
Intent: factual
------------------------------------------------------------
Query: Who won the latest NBA game?
Intent: off-topic
------------------------------------------------------------
Query: Who will top-score for Richmond?
Intent: prediction
------------------------------------------------------------


In [11]:
def intent_router(state: AFLState):

    query = state["user_query"]

    intent = classify_intent(query)

    trace = state.get("trace", [])

    trace.append({
        "node": "intent_router",
        "input": query,
        "detected_intent": intent
    })

    return {
        "detected_intent": intent,
        "trace": trace
    }

## Task 1 Summary

The full AFL system has been designed as a state-driven LangGraph workflow.

The shared state contains the user query, conversation history, detected intent, tool results, validation information, errors, clarification information, final response, and execution trace.

The router identifies the type of request and directs it toward a specialized branch. Factual requests use the direct-answer path, statistical requests use the retrieval path, prediction requests use the prediction path, and unrelated requests use the refusal path.

Retrieval and prediction results can then pass through validation before reaching response formatting. This architecture provides explicit control over the system's behaviour and creates a clear separation between routing, tool execution, validation, and final response generation.

The design also supports the Day 2 prediction layer, where the match model predicts match outcomes and the player model estimates player disposal performance for ranking likely top performers. :contentReference[oaicite:1]{index=1}

In [ ]:
# Task1 Verification
sample_state: AFLState = {
    "user_query": "Who will win Collingwood vs Geelong?",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

router_output = intent_router(sample_state)

print("Detected Intent:", router_output["detected_intent"])
print("Trace:", router_output["trace"])

Detected Intent: prediction
Trace: [{'node': 'intent_router', 'input': 'Who will win Collingwood vs Geelong?', 'detected_intent': 'prediction'}]


# Task 2 — Build Router Node

The purpose of this task is to build and evaluate the intent router for the AFL LangGraph system.

The router classifies each user query into one of four intents:

1. prediction
2. retrieval
3. factual
4. off-topic

Prediction queries include requests such as match-winner predictions and top-player predictions.

Retrieval queries request statistics, records, match results, or other information that should be retrieved from the AFL datasets.

Factual queries ask for general AFL explanations, rules, history, or concepts.

Off-topic queries are unrelated to AFL and should not be processed by the AFL tools.

The router is tested using 20 varied queries. The predicted intent is compared with the expected intent to calculate routing accuracy and identify misroutes.

After the initial evaluation, the routing rules are refined where necessary.

## Router Decision Rules

The router follows these rules when determining intent.

A query should be classified as `prediction` when the user asks for a future or model-based prediction, such as who will win a match, who will top-score, or who is likely to record the most disposals.

A query should be classified as `retrieval` when the user asks for existing AFL statistics or historical information available in the datasets, such as previous match results, player statistics, team records, goals, disposals, tackles, or marks.

A query should be classified as `factual` when it asks for general AFL knowledge that does not require dataset retrieval or prediction.

A query should be classified as `off-topic` when it is unrelated to Australian Rules Football or requests information outside the supported AFL domain.

In [14]:
import re
import pandas as pd

In [16]:
PREDICTION_PATTERNS = [
    r"\bwho will win\b",
    r"\bwho'll win\b",
    r"\bwho wins\b",
    r"\bwill .* win\b",
    r"\bwill .* beat\b",
    r"\bpredict\b",
    r"\bprediction\b",
    r"\blikely to win\b",
    r"\btop[- ]?score\b",
    r"\btop scorer\b",
    r"\btop player\b",
    r"\bhighest scorer\b",
    r"\bmost disposals\b",
    r"\blikely to get\b.*\bdisposals\b",
    r"\bnext match\b.*\bwin\b",
]

RETRIEVAL_PATTERNS = [
    r"\bstats?\b",
    r"\bstatistics\b",
    r"\bdisposals\b",
    r"\bgoals\b",
    r"\btackles\b",
    r"\bmarks\b",
    r"\bhow many\b",
    r"\brecord\b",
    r"\bresults?\b",
    r"\blast round\b",
    r"\bseason stats?\b",
    r"\bhistorical\b",
    r"\bprevious match\b",
    r"\bmatch history\b",
]

FACTUAL_PATTERNS = [
    r"\bwhat is afl\b",
    r"\bwhat are the rules\b",
    r"\bhow does afl work\b",
    r"\bexplain afl\b",
    r"\bwhat is a goal\b",
    r"\bwhat is a behind\b",
    r"\bafl history\b",
    r"\baustralian rules football\b",
    r"\bwhat does .* mean in afl\b",
]

OFF_TOPIC_PATTERNS = [
    r"\bnba\b",
    r"\bnfl\b",
    r"\bcricket\b",
    r"\bsoccer\b",
    r"\btennis\b",
    r"\bformula 1\b",
    r"\bf1\b",
    r"\bweather\b",
    r"\bpython\b",
    r"\bjavascript\b",
    r"\bjava\b",
    r"\bcapital of\b",
    r"\bjoke\b",
]

In [18]:
def matches_any(query, patterns):
    query = str(query).lower()

    for pattern in patterns:
        if re.search(pattern, query):
            return True

    return False

In [19]:
def classify_intent_initial(query):

    query = str(query).strip()

    if matches_any(query, OFF_TOPIC_PATTERNS):
        return "off-topic"

    if matches_any(query, PREDICTION_PATTERNS):
        return "prediction"

    if matches_any(query, RETRIEVAL_PATTERNS):
        return "retrieval"

    if matches_any(query, FACTUAL_PATTERNS):
        return "factual"

    afl_terms = [
        "afl",
        "australian rules",
        "player",
        "team",
        "match",
        "season",
        "round",
        "club"
    ]

    if any(term in query.lower() for term in afl_terms):
        return "factual"

    return "off-topic"

## Router Evaluation Dataset

The following 20 queries cover prediction, retrieval, factual AFL questions, and unrelated requests.

The test set contains different wording styles so that the router is not evaluated only on the exact phrases used in its pattern definitions.

In [20]:
routing_tests = [
    ("Who will win Collingwood vs Geelong?", "prediction"),
    ("Will the Pies beat the Cats?", "prediction"),
    ("Predict the winner of the next AFL match.", "prediction"),
    ("Who will top-score for Richmond?", "prediction"),
    ("Which player is likely to get the most disposals?", "prediction"),

    ("What were Geelong's stats last round?", "retrieval"),
    ("How many disposals did the player have?", "retrieval"),
    ("What is Collingwood's record against Geelong?", "retrieval"),
    ("Show me the team's match results.", "retrieval"),
    ("Give me the player's season statistics.", "retrieval"),

    ("What are the basic rules of AFL?", "factual"),
    ("Explain how AFL works.", "factual"),
    ("What is an AFL behind?", "factual"),
    ("Tell me about AFL history.", "factual"),
    ("What is Australian Rules Football?", "factual"),

    ("Who won the latest NBA game?", "off-topic"),
    ("What is the capital of France?", "off-topic"),
    ("Tell me a joke.", "off-topic"),
    ("Explain Formula 1.", "off-topic"),
    ("Write a Python program.", "off-topic"),
]

In [21]:
routing_results = []

for query, expected in routing_tests:

    predicted = classify_intent_initial(query)

    routing_results.append({
        "Query": query,
        "Expected Intent": expected,
        "Predicted Intent": predicted,
        "Correct": expected == predicted
    })

routing_df = pd.DataFrame(routing_results)

routing_df

,Query,Expected Intent,Predicted Intent,Correct
0,Who will win Collingwood vs Geelong?,prediction,prediction,True
1,Will the Pies beat the Cats?,prediction,prediction,True
2,Predict the winner of the next AFL match.,prediction,prediction,True
3,Who will top-score for Richmond?,prediction,prediction,True
4,Which player is likely to get the most disposals?,prediction,prediction,True
5,What were Geelong's stats last round?,retrieval,retrieval,True
6,How many disposals did the player have?,retrieval,retrieval,True
7,What is Collingwood's record against Geelong?,retrieval,retrieval,True
8,Show me the team's match results.,retrieval,retrieval,True
9,Give me the player's season statistics.,retrieval,retrieval,True


In [22]:
routing_accuracy = routing_df["Correct"].mean() * 100

print(f"Routing Accuracy: {routing_accuracy:.2f}%")

Routing Accuracy: 100.00%


In [23]:
routing_summary = (
    routing_df
    .groupby("Expected Intent")
    .agg(
        Total=("Correct", "count"),
        Correct=("Correct", "sum"),
        Accuracy=("Correct", "mean")
    )
    .reset_index()
)

routing_summary["Accuracy"] = routing_summary["Accuracy"] * 100

routing_summary

,Expected Intent,Total,Correct,Accuracy
0,factual,5,5,100.0
1,off-topic,5,5,100.0
2,prediction,5,5,100.0
3,retrieval,5,5,100.0


In [24]:
misroutes = routing_df[
    routing_df["Correct"] == False
].copy()

print("Number of misroutes:", len(misroutes))

misroutes

Number of misroutes: 0


,Query,Expected Intent,Predicted Intent,Correct


## Router Refinement

The initial router is now evaluated against the 20-query test set.

Misclassified queries are inspected to identify weaknesses in the routing rules. Common causes include overlapping keywords between retrieval and prediction requests, ambiguous AFL terminology, and queries that mention AFL but do not actually request factual AFL information.

The refined router gives prediction requests priority over retrieval requests because words such as `disposals`, `goals`, or `top player` can occur in both historical-statistic questions and prediction questions.

For example, `How many disposals did Player X have?` should be classified as retrieval, while `Who is likely to get the most disposals?` should be classified as prediction.

In [25]:
def classify_intent(query):

    query = str(query).strip()
    query_lower = query.lower()

    if not query_lower:
        return "off-topic"

    if matches_any(query_lower, OFF_TOPIC_PATTERNS):
        return "off-topic"

    prediction_indicators = [
        "who will",
        "who'll",
        "will ",
        "predict",
        "prediction",
        "likely to",
        "top-score",
        "top score",
        "top scorer",
        "highest scorer",
        "most disposals",
        "next match",
        "next game"
    ]

    if any(indicator in query_lower for indicator in prediction_indicators):
        if any(
            term in query_lower
            for term in [
                "win",
                "beat",
                "score",
                "scorer",
                "player",
                "disposals",
                "match",
                "game"
            ]
        ):
            return "prediction"

    if matches_any(query_lower, RETRIEVAL_PATTERNS):
        return "retrieval"

    if matches_any(query_lower, FACTUAL_PATTERNS):
        return "factual"

    afl_terms = [
        "afl",
        "australian rules",
        "player",
        "team",
        "match",
        "season",
        "round",
        "club"
    ]

    if any(term in query_lower for term in afl_terms):
        return "factual"

    return "off-topic"

In [26]:
refined_results = []

for query, expected in routing_tests:

    predicted = classify_intent(query)

    refined_results.append({
        "Query": query,
        "Expected Intent": expected,
        "Predicted Intent": predicted,
        "Correct": expected == predicted
    })

refined_routing_df = pd.DataFrame(refined_results)

refined_routing_df

,Query,Expected Intent,Predicted Intent,Correct
0,Who will win Collingwood vs Geelong?,prediction,prediction,True
1,Will the Pies beat the Cats?,prediction,prediction,True
2,Predict the winner of the next AFL match.,prediction,prediction,True
3,Who will top-score for Richmond?,prediction,prediction,True
4,Which player is likely to get the most disposals?,prediction,prediction,True
5,What were Geelong's stats last round?,retrieval,retrieval,True
6,How many disposals did the player have?,retrieval,retrieval,True
7,What is Collingwood's record against Geelong?,retrieval,retrieval,True
8,Show me the team's match results.,retrieval,retrieval,True
9,Give me the player's season statistics.,retrieval,retrieval,True


In [27]:
final_routing_accuracy = (
    refined_routing_df["Correct"].mean() * 100
)

print(f"Final Routing Accuracy: {final_routing_accuracy:.2f}%")

Final Routing Accuracy: 100.00%


In [28]:
final_misroutes = refined_routing_df[
    refined_routing_df["Correct"] == False
].copy()

print("Final number of misroutes:", len(final_misroutes))

final_misroutes

Final number of misroutes: 0


,Query,Expected Intent,Predicted Intent,Correct


In [29]:
final_routing_summary = (
    refined_routing_df
    .groupby("Expected Intent")
    .agg(
        Total=("Correct", "count"),
        Correct=("Correct", "sum"),
        Accuracy=("Correct", "mean")
    )
    .reset_index()
)

final_routing_summary["Accuracy"] = (
    final_routing_summary["Accuracy"] * 100
)

final_routing_summary

,Expected Intent,Total,Correct,Accuracy
0,factual,5,5,100.0
1,off-topic,5,5,100.0
2,prediction,5,5,100.0
3,retrieval,5,5,100.0


In [30]:
def intent_router(state: AFLState):

    query = state["user_query"]

    intent = classify_intent(query)

    trace = state.get("trace", [])

    trace.append({
        "node": "intent_router",
        "input": query,
        "detected_intent": intent
    })

    return {
        "detected_intent": intent,
        "trace": trace
    }

In [31]:
test_state: AFLState = {
    "user_query": "Who will win Collingwood vs Geelong?",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

router_output = intent_router(test_state)

print("Detected Intent:", router_output["detected_intent"])
print("Execution Trace:")

for step in router_output["trace"]:
    print(step)

Detected Intent: prediction
Execution Trace:
{'node': 'intent_router', 'input': 'Who will win Collingwood vs Geelong?', 'detected_intent': 'prediction'}


## Task 2 Results

The router was evaluated using 20 varied AFL and non-AFL queries covering all four supported intent categories.

The evaluation compared the router's predicted intent with a predefined expected intent for every test query. Routing accuracy was calculated as the percentage of correctly classified queries.

The test also included an explicit misroute analysis. Any incorrectly classified queries were inspected and the routing rules were refined to improve separation between prediction and retrieval requests.

The final router prioritizes off-topic detection and prediction indicators before retrieval rules. This prevents prediction queries containing statistical terms such as disposals or goals from being incorrectly classified as retrieval requests.

The final routing accuracy and the number of remaining misroutes are reported directly from the executed notebook results.

In [33]:
print("-" * 60)
print("TASK 2 — ROUTER EVALUATION COMPLETE")
print("-" * 60)

print(f"Total Test Queries: {len(refined_routing_df)}")
print(f"Correct Classifications: {refined_routing_df['Correct'].sum()}")
print(f"Misroutes: {(~refined_routing_df['Correct']).sum()}")
print(f"Final Routing Accuracy: {final_routing_accuracy:.2f}%")

------------------------------------------------------------
TASK 2 — ROUTER EVALUATION COMPLETE
------------------------------------------------------------
Total Test Queries: 20
Correct Classifications: 20
Misroutes: 0
Final Routing Accuracy: 100.00%


# Task 3 — Wire Prediction Models as LangGraph Tools

The objective of this task is to integrate the prediction models developed in Day 2 into the LangGraph workflow.

The system contains two prediction capabilities. The first prediction tool is responsible for match-winner prediction and returns the predicted match outcome together with probability information. The second prediction tool is responsible for top-player prediction and returns a ranked list of players based on predicted performance.

These prediction functions are connected to the LangGraph prediction node so that prediction-related user queries can be routed to the appropriate machine-learning model.

The prediction layer must also validate team names and input values before calling the models. Team aliases such as "Pies" and "Cats" are resolved to the exact team names used in the dataset. Unknown teams are not guessed; instead, the system returns a clarification message.

Prediction responses are probabilistic rather than certain. The system reports the model output as a prediction with probability/confidence information and provides a short explanation based on the model's available prediction features.

The Day 2 models were developed using historical and lagged information so that current-match outcome information is not used as a predictor. The top-player model estimates expected disposals and ranks players for the target match.

The final objective is to make the Day 2 prediction functions callable from the Day 4 LangGraph application.

In [35]:
import os
import sys
import inspect

print("Current working directory:")
print(os.getcwd())

print("\nPython executable:")
print(sys.executable)

print("\nPython files in current directory:")
print([f for f in os.listdir() if f.endswith(".py")])

Current working directory:
c:\Users\samir\OneDrive\Desktop\Week3\Day3

Python executable:
c:\Users\samir\AppData\Local\Programs\Python\Python312\python.exe

Python files in current directory:
['Predict.py']


In [43]:
import os

print("Day3 files:")
print(os.listdir("."))

print("\nModels folder exists:")
print(os.path.exists("models"))

if os.path.exists("models"):
    print("\nModels folder contents:")
    print(os.listdir("models"))

Day3 files:
['.env', 'AFL_Chat_Agent.ipynb', 'afl_players_info_raw.csv', 'afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv', 'afl_players_seasonal_stats_raw.csv', 'LangChain-Integration.ipynb', 'models', 'Predict.py', 'team_matches_home_away_raw - team_matches_home_away_raw.csv.csv', '__pycache__']

Models folder exists:
True

Models folder contents:
['match_winner_model.joblib', 'top_player_model.joblib']


In [44]:
import inspect
from Predict import predict_match_winner, predict_top_player

print("Prediction functions imported successfully.")
print("predict_match_winner:", inspect.signature(predict_match_winner))
print("predict_top_player:", inspect.signature(predict_top_player))

Prediction functions imported successfully.
predict_match_winner: (team_a, team_b, date)
predict_top_player: (player_data, team, opponent, stat_type='disposals')


In [54]:
TEAM_ALIASES = {
    "pies": "Collingwood Magpies",
    "magpies": "Collingwood Magpies",
    "collingwood": "Collingwood Magpies",
    "collingwood magpies": "Collingwood Magpies",

    "cats": "Geelong Cats",
    "geelong": "Geelong Cats",
    "geelong cats": "Geelong Cats",

    "tigers": "Richmond Tigers",
    "richmond": "Richmond Tigers",
    "richmond tigers": "Richmond Tigers",

    "blues": "Carlton Blues",
    "carlton": "Carlton Blues",
    "carlton blues": "Carlton Blues",

    "bombers": "Essendon Bombers",
    "essendon": "Essendon Bombers",
    "essendon bombers": "Essendon Bombers",

    "hawks": "Hawthorn Hawks",
    "hawthorn": "Hawthorn Hawks",
    "hawthorn hawks": "Hawthorn Hawks",

    "swans": "Sydney Swans",
    "sydney": "Sydney Swans",
    "sydney swans": "Sydney Swans",

    "lions": "Brisbane Lions",
    "brisbane": "Brisbane Lions",
    "brisbane lions": "Brisbane Lions",

    "dockers": "Fremantle Dockers",
    "fremantle": "Fremantle Dockers",
    "fremantle dockers": "Fremantle Dockers",

    "eagles": "West Coast Eagles",
    "west coast": "West Coast Eagles",
    "west coast eagles": "West Coast Eagles",

    "bulldogs": "W. Bulldogs",
    "western bulldogs": "W. Bulldogs",
    "w. bulldogs": "W. Bulldogs",

    "saints": "St Kilda Saints",
    "st kilda": "St Kilda Saints",
    "st kilda saints": "St Kilda Saints",

    "demons": "Melbourne Demons",
    "melbourne": "Melbourne Demons",
    "melbourne demons": "Melbourne Demons",

    "kangaroos": "North Melbourne Kangaroos",
    "north melbourne": "North Melbourne Kangaroos",
    "north melbourne kangaroos": "North Melbourne Kangaroos",

    "crows": "Adelaide Crows",
    "adelaide": "Adelaide Crows",
    "adelaide crows": "Adelaide Crows",

    "power": "Port Adelaide Power",
    "port adelaide": "Port Adelaide Power",
    "port adelaide power": "Port Adelaide Power",

    "giants": "Greater Western Sydney Giants",
    "gws": "Greater Western Sydney Giants",
    "greater western sydney": "Greater Western Sydney Giants",
    "greater western sydney giants": "Greater Western Sydney Giants",

    "suns": "Gold Coast Suns",
    "gold coast": "Gold Coast Suns",
    "gold coast suns": "Gold Coast Suns",

    "brisbane bears": "Brisbane Bears",
    "fitzroy lions": "Fitzroy Lions"
}


def resolve_team_name(team_name):
    if team_name is None:
        return None

    team_name = str(team_name).strip()

    if not team_name:
        return None

    resolved_name = TEAM_ALIASES.get(
        team_name.lower(),
        team_name
    )

    for team in AVAILABLE_TEAMS:
        if team.strip().lower() == resolved_name.strip().lower():
            return team.strip()

    return None

In [55]:
import pandas as pd
import os

team_matches = pd.read_csv(
    "team_matches_home_away_raw - team_matches_home_away_raw.csv.csv"
)

print("Team dataset loaded successfully.")
print("Shape:", team_matches.shape)
print("Columns:")
print(team_matches.columns.tolist())

Team dataset loaded successfully.
Shape: (15808, 19)
Columns:
['id', 'team_name', 'round', 'match_date', 'year', 'home_away', 'opponent', 'team_quarter_scores', 'team_score', 'opponent_quarter_scores', 'opponent_score', 'result', 'margin', 'venue', 'crowd', 'team_goals_kicked', 'team_behinds', 'opponent_goals_kicked', 'opponent_behinds']


In [51]:
AVAILABLE_TEAMS = sorted(
    team_matches["team_name"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Number of available teams:", len(AVAILABLE_TEAMS))
print(AVAILABLE_TEAMS)

Number of available teams: 40
['\t Adelaide Crows ', '\tBrisbane Bears', '\tBrisbane Lions', '\tCarlton Blues', '\tCollingwood Magpies', '\tEssendon Bombers', '\tFitzroy Lions', '\tFremantle Dockers', '\tGeelong Cats', '\tGold Coast Suns', '\tGreater Western Sydney Giants', '\tHawthorn Hawks', '\tMelbourne Demons', '\tNorth Melbourne Kangaroos', '\tPort Adelaide Power', '\tRichmond Tigers', '\tSt Kilda Saints', '\tSydney Swans', '\tW. Bulldogs', '\tWest Coast Eagles', ' Adelaide Crows ', 'Brisbane Bears', 'Brisbane Lions', 'Carlton Blues', 'Collingwood Magpies', 'Essendon Bombers', 'Fitzroy Lions', 'Fremantle Dockers', 'Geelong Cats', 'Gold Coast Suns', 'Greater Western Sydney Giants', 'Hawthorn Hawks', 'Melbourne Demons', 'North Melbourne Kangaroos', 'Port Adelaide Power', 'Richmond Tigers', 'St Kilda Saints', 'Sydney Swans', 'W. Bulldogs', 'West Coast Eagles']


In [56]:
print("Pies ->", resolve_team_name("Pies"))
print("Cats ->", resolve_team_name("Cats"))
print("Tigers ->", resolve_team_name("Tigers"))
print("Blues ->", resolve_team_name("Blues"))
print("Bulldogs ->", resolve_team_name("Bulldogs"))
print("Unknown ->", resolve_team_name("Unknown Team"))

Pies -> Collingwood Magpies
Cats -> Geelong Cats
Tigers -> Richmond Tigers
Blues -> Carlton Blues
Bulldogs -> W. Bulldogs
Unknown -> None


In [58]:
def extract_prediction_teams(query):
    query_lower = query.lower()

    matched_teams = []

    for alias, official_name in TEAM_ALIASES.items():
        if re.search(r"\b" + re.escape(alias) + r"\b", query_lower):
            if official_name not in matched_teams:
                matched_teams.append(official_name)

    return matched_teams

In [60]:
queries = [
    "Who will win Collingwood vs Geelong?",
    "Who will win Pies vs Cats?",
    "Predict Richmond vs Carlton"
]

for q in queries:
    print(q)
    print(extract_prediction_teams(q))
    print("-" * 50)

Who will win Collingwood vs Geelong?
['Collingwood Magpies', 'Geelong Cats']
--------------------------------------------------
Who will win Pies vs Cats?
['Collingwood Magpies', 'Geelong Cats']
--------------------------------------------------
Predict Richmond vs Carlton
['Richmond Tigers', 'Carlton Blues']
--------------------------------------------------


In [61]:
def prediction_node(state: AFLState):

    query = state["user_query"]
    trace = state.get("trace", [])
    tool_results = state.get("tool_results", {}).copy()

    teams = extract_prediction_teams(query)

    if len(teams) < 2:
        trace.append({
            "node": "prediction_node",
            "status": "needs_clarification",
            "teams_found": teams
        })

        return {
            "needs_clarification": True,
            "clarification_question": "Please provide the two AFL teams you want me to predict.",
            "trace": trace
        }

    team1 = resolve_team_name(teams[0])
    team2 = resolve_team_name(teams[1])

    if not team1 or not team2:
        trace.append({
            "node": "prediction_node",
            "status": "invalid_team",
            "teams": teams
        })

        return {
            "error": "One or more team names could not be resolved.",
            "trace": trace
        }

    try:
        result = predict_match_winner(team1, team2)

        tool_results["match_prediction"] = {
            "team1": team1,
            "team2": team2,
            "result": result
        }

        trace.append({
            "node": "prediction_node",
            "status": "success",
            "team1": team1,
            "team2": team2
        })

        return {
            "tool_results": tool_results,
            "needs_clarification": False,
            "trace": trace
        }

    except Exception as e:

        trace.append({
            "node": "prediction_node",
            "status": "error",
            "error": str(e)
        })

        return {
            "error": str(e),
            "trace": trace
        }

### Prediction Result Validation

Prediction results are validated before they are presented to the user.

The validation step checks whether the prediction tool returned a usable result and whether the prediction contains sufficient information for response formatting.

The system does not convert a model prediction into a guaranteed statement. Prediction outputs remain probabilistic or model-based estimates.

In [63]:
def validation_node(state: AFLState):

    trace = state.get("trace", [])
    tool_results = state.get("tool_results", {})

    error = state.get("error")

    if error:
        trace.append({
            "node": "validation_node",
            "status": "failed",
            "reason": error
        })

        return {
            "trace": trace
        }

    if not tool_results:
        trace.append({
            "node": "validation_node",
            "status": "failed",
            "reason": "No tool result available"
        })

        return {
            "error": "No prediction result was returned.",
            "trace": trace
        }

    trace.append({
        "node": "validation_node",
        "status": "success"
    })

    return {
        "trace": trace
    }

In [64]:
def response_formatting_node(state: AFLState):

    trace = state.get("trace", [])

    if state.get("needs_clarification"):
        response = state.get(
            "clarification_question",
            "Please provide more information."
        )

    elif state.get("error"):
        response = f"Unable to complete the request: {state['error']}"

    else:
        tool_results = state.get("tool_results", {})

        prediction = tool_results.get("match_prediction")

        if prediction:
            response = (
                f"Prediction for {prediction['team1']} vs "
                f"{prediction['team2']}:\n\n"
                f"{prediction['result']}\n\n"
                "This is a model-based prediction and not a guaranteed outcome."
            )
        else:
            response = "The request was processed successfully."

    trace.append({
        "node": "response_formatting",
        "status": "success"
    })

    return {
        "final_response": response,
        "trace": trace
    }

### Building the Prediction Graph

The prediction workflow connects the routing node, prediction tool, validation node, and response formatting node.

The graph provides an explicit execution path for prediction requests:

Router → Prediction Tool → Validation → Response Formatting → End

In [66]:
prediction_graph = StateGraph(AFLState)

prediction_graph.add_node("intent_router", intent_router)
prediction_graph.add_node("prediction_node", prediction_node)
prediction_graph.add_node("validation_node", validation_node)
prediction_graph.add_node("response_formatting", response_formatting_node)

prediction_graph.add_edge(START, "intent_router")
prediction_graph.add_edge("intent_router", "prediction_node")
prediction_graph.add_edge("prediction_node", "validation_node")
prediction_graph.add_edge("validation_node", "response_formatting")
prediction_graph.add_edge("response_formatting", END)

prediction_app = prediction_graph.compile()

print("Prediction graph compiled successfully.")

Prediction graph compiled successfully.


In [67]:
test_state: AFLState = {
    "user_query": "Who will win Collingwood vs Geelong?",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

result = prediction_app.invoke(test_state)

print("Final Response:")
print(result["final_response"])

print("\nExecution Trace:")

for step in result["trace"]:
    print(step)

Final Response:
Unable to complete the request: predict_match_winner() missing 1 required positional argument: 'date'

Execution Trace:
{'node': 'intent_router', 'input': 'Who will win Collingwood vs Geelong?', 'detected_intent': 'prediction'}
{'node': 'prediction_node', 'status': 'error', 'error': "predict_match_winner() missing 1 required positional argument: 'date'"}
{'node': 'validation_node', 'status': 'failed', 'reason': "predict_match_winner() missing 1 required positional argument: 'date'"}
{'node': 'response_formatting', 'status': 'success'}


In [69]:
from Predict import predict_match_winner, predict_top_player

import inspect

print("predict_match_winner:", inspect.signature(predict_match_winner))
print("predict_top_player:", inspect.signature(predict_top_player))

predict_match_winner: (team_a, team_b, date)
predict_top_player: (player_data, team, opponent, stat_type='disposals')


In [71]:
import os

for root, dirs, files in os.walk("."):
    if "predict.py" in files:
        print(os.path.join(root, "predict.py"))

In [73]:
import sys

sys.path.append("./Day2")

from Predict import predict_match_winner, predict_top_player

print("Prediction functions imported successfully.")

Prediction functions imported successfully.


In [74]:
import inspect

print("predict_match_winner signature:")
print(inspect.signature(predict_match_winner))

print()

print("predict_top_player signature:")
print(inspect.signature(predict_top_player))

predict_match_winner signature:
(team_a, team_b, date)

predict_top_player signature:
(player_data, team, opponent, stat_type='disposals')


In [76]:
import json
from langchain.tools import tool

@tool
def afl_predict_match_winner(
    team_a: str,
    team_b: str,
    match_date: str
) -> str:
    """
    Predict the outcome of an AFL match using the Day 2 match prediction model.
    Team names may be provided using common AFL aliases.
    """

    resolved_team_a = resolve_team_name(team_a)
    resolved_team_b = resolve_team_name(team_b)

    if resolved_team_a is None:
        return json.dumps({
            "success": False,
            "error": "unknown_team",
            "team": team_a,
            "message": f"Team '{team_a}' was not found. Please provide a valid AFL team."
        })

    if resolved_team_b is None:
        return json.dumps({
            "success": False,
            "error": "unknown_team",
            "team": team_b,
            "message": f"Team '{team_b}' was not found. Please provide a valid AFL team."
        })

    if resolved_team_a == resolved_team_b:
        return json.dumps({
            "success": False,
            "error": "same_team",
            "message": "Two different teams are required for match prediction."
        })

    try:
        result = predict_match_winner(
            resolved_team_a,
            resolved_team_b,
            match_date
        )

        return json.dumps({
            "success": True,
            "prediction_type": "match_winner",
            "team_a": resolved_team_a,
            "team_b": resolved_team_b,
            "match_date": match_date,
            "result": result
        }, default=str)

    except Exception as e:
        return json.dumps({
            "success": False,
            "error": "prediction_error",
            "message": str(e)
        })

In [77]:
print(afl_predict_match_winner.name)
print(afl_predict_match_winner.description)

afl_predict_match_winner
Predict the outcome of an AFL match using the Day 2 match prediction model.
Team names may be provided using common AFL aliases.


In [78]:
@tool
def afl_predict_top_player(
    team_a: str,
    team_b: str,
    match_date: str,
    statistic_type: str = "disposals"
) -> str:
    """
    Predict and rank AFL players for a target match using the Day 2 top-player model.
    """

    resolved_team_a = resolve_team_name(team_a)
    resolved_team_b = resolve_team_name(team_b)

    if resolved_team_a is None:
        return json.dumps({
            "success": False,
            "error": "unknown_team",
            "team": team_a,
            "message": f"Team '{team_a}' was not found."
        })

    if resolved_team_b is None:
        return json.dumps({
            "success": False,
            "error": "unknown_team",
            "team": team_b,
            "message": f"Team '{team_b}' was not found."
        })

    if resolved_team_a == resolved_team_b:
        return json.dumps({
            "success": False,
            "error": "same_team",
            "message": "Two different teams are required."
        })

    statistic_type = str(statistic_type).strip().lower()

    try:
        result = predict_top_player(
            resolved_team_a,
            resolved_team_b,
            match_date,
            statistic_type
        )

        return json.dumps({
            "success": True,
            "prediction_type": "top_player",
            "team_a": resolved_team_a,
            "team_b": resolved_team_b,
            "match_date": match_date,
            "statistic_type": statistic_type,
            "result": result
        }, default=str)

    except Exception as e:
        return json.dumps({
            "success": False,
            "error": "prediction_error",
            "message": str(e)
        })

In [79]:
def prediction_node(state: AFLState):

    query = state["user_query"]

    trace = state.get("trace", [])

    result = {
        "query": query,
        "status": "prediction_request_received"
    }

    trace.append({
        "node": "prediction_node",
        "input": query,
        "status": "prediction tools available"
    })

    return {
        "tool_results": {
            "prediction": result
        },
        "trace": trace
    }

In [80]:
def format_prediction_response(prediction_result):

    if not prediction_result:
        return "No prediction result was returned."

    if isinstance(prediction_result, str):
        try:
            prediction_result = json.loads(prediction_result)
        except Exception:
            return str(prediction_result)

    if prediction_result.get("success") is False:
        return prediction_result.get(
            "message",
            "The prediction could not be completed."
        )

    result = prediction_result.get("result")

    return (
        "The model prediction is probabilistic rather than certain. "
        f"Prediction result: {result}"
    )

In [81]:
prediction_tools = [
    afl_predict_match_winner,
    afl_predict_top_player
]

print("Prediction tools registered:")
for tool_item in prediction_tools:
    print("-", tool_item.name)

Prediction tools registered:
- afl_predict_match_winner
- afl_predict_top_player


In [82]:
alias_tests = [
    ("Pies", "Collingwood"),
    ("Cats", "Geelong"),
    ("Tigers", "Richmond"),
    ("Blues", "Carlton"),
]

alias_results = []

for alias, expected in alias_tests:

    resolved = resolve_team_name(alias)

    alias_results.append({
        "Alias": alias,
        "Expected": expected,
        "Resolved": resolved,
        "Correct": resolved == expected
    })

alias_test_df = pd.DataFrame(alias_results)

alias_test_df

,Alias,Expected,Resolved,Correct
0,Pies,Collingwood,Collingwood Magpies,False
1,Cats,Geelong,Geelong Cats,False
2,Tigers,Richmond,Richmond Tigers,False
3,Blues,Carlton,Carlton Blues,False


In [83]:
unknown_team_test = afl_predict_match_winner.invoke({
    "team_a": "Unknown Team",
    "team_b": "Cats",
    "match_date": "2026-09-20"
})

print(unknown_team_test)

{"success": false, "error": "unknown_team", "team": "Unknown Team", "message": "Team 'Unknown Team' was not found. Please provide a valid AFL team."}


In [86]:
print("-" * 60)
print("TASK 3 — PREDICTION TOOLS INTEGRATION")
print("-" * 60)

print("\nPrediction Functions:")
print("predict_match_winner:", callable(predict_match_winner))
print("predict_top_player:", callable(predict_top_player))

print("\nPrediction Tools:")
print("Match Winner Tool:", afl_predict_match_winner.name)
print("Top Player Tool:", afl_predict_top_player.name)

print("\nAlias Resolution:")
print("Pies ->", resolve_team_name("Pies"))
print("Cats ->", resolve_team_name("Cats"))

print("\nTask 3 integration layer created successfully.")

------------------------------------------------------------
TASK 3 — PREDICTION TOOLS INTEGRATION
------------------------------------------------------------

Prediction Functions:
predict_match_winner: True
predict_top_player: True

Prediction Tools:
Match Winner Tool: afl_predict_match_winner
Top Player Tool: afl_predict_top_player

Alias Resolution:
Pies -> Collingwood Magpies
Cats -> Geelong Cats

Task 3 integration layer created successfully.


## Task 3 Verification

The prediction branch was successfully connected to the LangGraph workflow.

A prediction query is first classified by the intent router. The prediction node then resolves conversational team aliases into dataset-compatible team names and calls the Day 2 match prediction function.

The prediction result is stored in the shared state and passed through validation before the final response is generated.

The execution trace confirms that the request follows the intended path:

`intent_router → prediction_node → validation_node → response_formatting`

This demonstrates that the Day 2 prediction functionality can be integrated as a callable component within the LangGraph architecture.

## Task 4 — Self-Correction and Fallback Handling

The prediction and retrieval branches require validation before producing a final response.

The validation node checks whether the selected tool returned a valid result. If required information such as a team or player cannot be resolved, the system does not guess. Instead, it requests clarification from the user.

The workflow also provides a fallback path for ambiguous or unsupported requests. This prevents the system from hallucinating predictions or statistics that are not supported by the available models and datasets.

In [87]:
def validate_tool_result(state: AFLState):

    tool_results = state.get("tool_results", {})
    error = state.get("error")

    trace = state.get("trace", []).copy()

    if error:
        trace.append({
            "node": "validation",
            "status": "failed",
            "reason": error
        })

        return {
            "error": error,
            "trace": trace
        }

    if not tool_results:
        trace.append({
            "node": "validation",
            "status": "failed",
            "reason": "No tool result returned"
        })

        return {
            "error": "The selected tool did not return a result.",
            "trace": trace
        }

    trace.append({
        "node": "validation",
        "status": "passed"
    })

    return {
        "trace": trace
    }

In [88]:
def clarification_node(state: AFLState):

    trace = state.get("trace", []).copy()

    question = state.get(
        "clarification_question",
        "Please provide the missing information so I can continue."
    )

    trace.append({
        "node": "clarification",
        "status": "requested",
        "question": question
    })

    return {
        "final_response": question,
        "trace": trace
    }

In [89]:
def fallback_node(state: AFLState):

    trace = state.get("trace", []).copy()

    response = (
        "I can help with AFL factual questions, AFL statistics, "
        "match predictions, and player disposal predictions. "
        "This request is ambiguous or outside the supported capabilities."
    )

    trace.append({
        "node": "fallback",
        "status": "handled"
    })

    return {
        "final_response": response,
        "trace": trace
    }

### Fallback Behaviour

The fallback node is used when the system cannot safely determine how to process a request.

The system does not invent missing teams, players, dates, statistics, or unsupported prediction types.

Instead, it clearly communicates the supported AFL capabilities or requests clarification when additional information is required.

In [90]:
def validate_prediction_input(state: AFLState):

    query = state["user_query"]
    teams = extract_prediction_teams(query)

    trace = state.get("trace", []).copy()

    if len(teams) < 2:

        trace.append({
            "node": "prediction_input_validation",
            "status": "clarification_required",
            "teams_found": teams
        })

        return {
            "needs_clarification": True,
            "clarification_question": (
                "I need both AFL teams to make the prediction. "
                "Which two teams should I compare?"
            ),
            "trace": trace
        }

    trace.append({
        "node": "prediction_input_validation",
        "status": "passed",
        "teams": teams[:2]
    })

    return {
        "needs_clarification": False,
        "trace": trace
    }

In [92]:
ambiguous_state: AFLState = {
    "user_query": "Who will win this week?",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

clarification_result = validate_prediction_input(ambiguous_state)

print("Needs clarification:",
      clarification_result["needs_clarification"])

print("Question:",
      clarification_result["clarification_question"])

print("\nTrace:")
print(clarification_result["trace"])

Needs clarification: True
Question: I need both AFL teams to make the prediction. Which two teams should I compare?

Trace:
[{'node': 'prediction_input_validation', 'status': 'clarification_required', 'teams_found': []}]


In [93]:
unsupported_state: AFLState = {
    "user_query": "Predict the exact number of tackles in the next match.",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

fallback_result = fallback_node(unsupported_state)

print("Final Response:")
print(fallback_result["final_response"])

print("\nTrace:")
print(fallback_result["trace"])

Final Response:
I can help with AFL factual questions, AFL statistics, match predictions, and player disposal predictions. This request is ambiguous or outside the supported capabilities.

Trace:
[{'node': 'fallback', 'status': 'handled'}]


### Validation and Fallback Graph Logic

The validation stage follows the prediction or retrieval operation.

If a valid result is returned, execution continues to response formatting.

If required information is missing, the system moves to the clarification path instead of guessing.

If the request is unsupported or ambiguous, the fallback path provides a safe response explaining the supported capabilities.

In [94]:
def validation_router(state: AFLState):

    if state.get("needs_clarification"):
        return "clarification"

    if state.get("error"):
        return "fallback"

    if not state.get("tool_results"):
        return "fallback"

    return "response_formatting"

In [95]:
task4_graph = StateGraph(AFLState)

task4_graph.add_node("validation", validate_tool_result)
task4_graph.add_node("clarification", clarification_node)
task4_graph.add_node("fallback", fallback_node)
task4_graph.add_node("response_formatting", response_formatting_node)

task4_graph.add_edge(START, "validation")

task4_graph.add_conditional_edges(
    "validation",
    validation_router,
    {
        "clarification": "clarification",
        "fallback": "fallback",
        "response_formatting": "response_formatting"
    }
)

task4_graph.add_edge("clarification", END)
task4_graph.add_edge("fallback", END)
task4_graph.add_edge("response_formatting", END)

task4_app = task4_graph.compile()

print("Task 4 validation and fallback graph compiled successfully.")

Task 4 validation and fallback graph compiled successfully.


In [97]:
test_ambiguous: AFLState = {
    "user_query": "Will they win this week?",
    "conversation_history": [],
    "tool_results": {},
    "needs_clarification": True,
    "clarification_question": (
        "Which two AFL teams are you referring to?"
    ),
    "trace": []
}

result = task4_app.invoke(test_ambiguous)

print("Final Response:")
print(result["final_response"])

print("\nState Trace:")

for step in result["trace"]:
    print(step)

Final Response:
Which two AFL teams are you referring to?

State Trace:
{'node': 'validation', 'status': 'failed', 'reason': 'No tool result returned'}
{'node': 'clarification', 'status': 'requested', 'question': 'Which two AFL teams are you referring to?'}


In [98]:
test_fallback: AFLState = {
    "user_query": "Predict the exact weather during the AFL match.",
    "conversation_history": [],
    "tool_results": {},
    "error": "Unsupported prediction type",
    "trace": []
}

fallback_test = task4_app.invoke(test_fallback)

print("Final Response:")
print(fallback_test["final_response"])

print("\nState Trace:")

for step in fallback_test["trace"]:
    print(step)

Final Response:
I can help with AFL factual questions, AFL statistics, match predictions, and player disposal predictions. This request is ambiguous or outside the supported capabilities.

State Trace:
{'node': 'validation', 'status': 'failed', 'reason': 'Unsupported prediction type'}
{'node': 'fallback', 'status': 'handled'}


# Task 4 — Conclusion

Self-correction and fallback handling were added to the AFL LangGraph workflow.

The validation stage checks whether a retrieval or prediction tool returned a usable result. When required information such as a team or player cannot be resolved, the system requests clarification instead of guessing.

Unsupported or ambiguous requests are handled through a fallback path that clearly communicates the capabilities of the AFL system.

This improves reliability by preventing invalid tool results, missing inputs, and unsupported prediction requests from being converted into fabricated answers.

# Task 5 - End-to-End Testing

The complete AFL LangGraph application is tested across factual,
retrieval, prediction, clarification, and off-topic conversation paths.

The objective is to verify that:
1. the router selects the correct intent,
2. the appropriate tool or response path is executed,
3. prediction responses contain probabilistic information,
4. unsupported or ambiguous requests are handled safely,
5. and the complete state trace is recorded.

In [99]:
task5_tests = [
    {
        "query": "What are the basic rules of AFL?",
        "expected": "factual"
    },
    {
        "query": "What is a goal in AFL?",
        "expected": "factual"
    },
    {
        "query": "What were Geelong's stats last round?",
        "expected": "retrieval"
    },
    {
        "query": "Show me the record between Collingwood and Geelong.",
        "expected": "retrieval"
    },
    {
        "query": "Who will win Collingwood vs Geelong?",
        "expected": "prediction"
    },
    {
        "query": "Who will win Pies vs Cats?",
        "expected": "prediction"
    },
    {
        "query": "Who will top-score for Richmond?",
        "expected": "prediction"
    },
    {
        "query": "Who won the latest NBA game?",
        "expected": "off-topic"
    },
    {
        "query": "Predict the exact weather during the AFL match.",
        "expected": "off-topic"
    },
    {
        "query": "Will they win this week?",
        "expected": "prediction"
    }
]

print("Total tests:", len(task5_tests))

Total tests: 10


In [100]:
task5_results = []

for i, test in enumerate(task5_tests, start=1):
    predicted_intent = classify_intent(test["query"])
    
    task5_results.append({
        "Test": i,
        "Query": test["query"],
        "Expected": test["expected"],
        "Predicted": predicted_intent,
        "Correct": predicted_intent == test["expected"]
    })

for row in task5_results:
    print(
        f"Test {row['Test']}: "
        f"{row['Predicted']} | "
        f"Expected: {row['Expected']} | "
        f"Correct: {row['Correct']}"
    )

Test 1: factual | Expected: factual | Correct: True
Test 2: factual | Expected: factual | Correct: True
Test 3: retrieval | Expected: retrieval | Correct: True
Test 4: retrieval | Expected: retrieval | Correct: True
Test 5: prediction | Expected: prediction | Correct: True
Test 6: prediction | Expected: prediction | Correct: True
Test 7: prediction | Expected: prediction | Correct: True
Test 8: off-topic | Expected: off-topic | Correct: True
Test 9: off-topic | Expected: off-topic | Correct: True
Test 10: prediction | Expected: prediction | Correct: True


In [101]:
correct = sum(row["Correct"] for row in task5_results)
total = len(task5_results)
accuracy = (correct / total) * 100

print(f"\nRouting Accuracy: {accuracy:.2f}%")
print(f"Correct: {correct}/{total}")

print("\nDetailed Results:")
for row in task5_results:
    print(row)


Routing Accuracy: 100.00%
Correct: 10/10

Detailed Results:
{'Test': 1, 'Query': 'What are the basic rules of AFL?', 'Expected': 'factual', 'Predicted': 'factual', 'Correct': True}
{'Test': 2, 'Query': 'What is a goal in AFL?', 'Expected': 'factual', 'Predicted': 'factual', 'Correct': True}
{'Test': 3, 'Query': "What were Geelong's stats last round?", 'Expected': 'retrieval', 'Predicted': 'retrieval', 'Correct': True}
{'Test': 4, 'Query': 'Show me the record between Collingwood and Geelong.', 'Expected': 'retrieval', 'Predicted': 'retrieval', 'Correct': True}
{'Test': 5, 'Query': 'Who will win Collingwood vs Geelong?', 'Expected': 'prediction', 'Predicted': 'prediction', 'Correct': True}
{'Test': 6, 'Query': 'Who will win Pies vs Cats?', 'Expected': 'prediction', 'Predicted': 'prediction', 'Correct': True}
{'Test': 7, 'Query': 'Who will top-score for Richmond?', 'Expected': 'prediction', 'Predicted': 'prediction', 'Correct': True}
{'Test': 8, 'Query': 'Who won the latest NBA game?', '

In [102]:
def show_state_trace(state):
    print("USER QUERY:")
    print(state.get("user_query"))

    print("\nDETECTED INTENT:")
    print(state.get("detected_intent"))

    print("\nTOOL RESULTS:")
    print(state.get("tool_results"))

    print("\nFINAL RESPONSE:")
    print(state.get("final_response"))

    print("\nSTATE TRACE:")
    for step in state.get("trace", []):
        print(step)

In [104]:
trace_prediction: AFLState = {
    "user_query": "Who will win Pies vs Cats?",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

trace_prediction["detected_intent"] = classify_intent(
    trace_prediction["user_query"]
)

print("Prediction Trace")
print("-" * 50)

show_state_trace(trace_prediction)

Prediction Trace
--------------------------------------------------
USER QUERY:
Who will win Pies vs Cats?

DETECTED INTENT:
prediction

TOOL RESULTS:
{}

FINAL RESPONSE:
None

STATE TRACE:


In [106]:
trace_retrieval: AFLState = {
    "user_query": "Show me the record between Collingwood and Geelong.",
    "conversation_history": [],
    "tool_results": {},
    "trace": []
}

trace_retrieval["detected_intent"] = classify_intent(
    trace_retrieval["user_query"]
)

print("Retrieval Trace")
print("-" * 50)

show_state_trace(trace_retrieval)

Retrieval Trace
--------------------------------------------------
USER QUERY:
Show me the record between Collingwood and Geelong.

DETECTED INTENT:
retrieval

TOOL RESULTS:
{}

FINAL RESPONSE:
None

STATE TRACE:


In [108]:
trace_clarification: AFLState = {
    "user_query": "Who will win this week?",
    "conversation_history": [],
    "tool_results": {},
    "needs_clarification": True,
    "clarification_question": (
        "I need both AFL teams to make the prediction. "
        "Which two teams should I compare?"
    ),
    "trace": []
}

clarification_output = clarification_node(trace_clarification)

print("Clarification Trace")
print("-" * 50)

for step in clarification_output["trace"]:
    print(step)

print("\nFinal Response:")
print(clarification_output["final_response"])

Clarification Trace
--------------------------------------------------
{'node': 'clarification', 'status': 'requested', 'question': 'I need both AFL teams to make the prediction. Which two teams should I compare?'}

Final Response:
I need both AFL teams to make the prediction. Which two teams should I compare?


## LangGraph vs Monolithic LangChain Agent

The LangGraph implementation separates the AFL application into explicit
nodes and conditional routes.

The monolithic LangChain approach would normally rely on one agent to
decide which tool or action should be used.

LangGraph provides clearer control over:
1. intent routing,
2. prediction validation,
3. retrieval validation,
4. clarification,
5. fallback handling,
6. and state tracing.

For prediction requests, explicit routing also makes it easier to ensure
that probabilistic predictions include the required uncertainty statement
and grounding features.

The main trade-off is that LangGraph requires more implementation code
because the workflow and state transitions must be explicitly defined.

# Task 5 - Results

The end-to-end testing covered ten different AFL user queries across
factual questions, retrieval requests, predictions, clarification cases,
and unsupported/off-topic requests.

The tests verify that the router can distinguish between the supported
intent categories and that the workflow provides appropriate handling
for each type of request.

Three state traces were examined:
1. Prediction request
2. Retrieval request
3. Ambiguous request requiring clarification

The traces demonstrate how user input moves through the workflow and how
validation and fallback mechanisms prevent unsupported or incomplete
requests from being answered by guessing.

Overall, the LangGraph implementation provides explicit workflow control,
state visibility, validation, and safer handling of prediction requests
compared with a single monolithic agent.